<a href="https://colab.research.google.com/github/KeerthanaSistla/BigDataAnalytics/blob/main/BDA_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BDA Assignment") \
    .getOrCreate()

spark

In [3]:
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



PART 1: HIVE PARTITIONING

In [4]:
data = [
    (1, "Keerthana", "Math", 85, "Sem1"),
    (2, "Rahul", "Physics", 78, "Sem1"),
    (3, "Anjali", "Chemistry", 92, "Sem2"),
    (4, "Arjun", "Math", 88, "Sem2")
]

columns = ["student_id", "name", "subject", "marks", "semester"]

df = spark.createDataFrame(data, columns)
df.show()

+----------+---------+---------+-----+--------+
|student_id|     name|  subject|marks|semester|
+----------+---------+---------+-----+--------+
|         1|Keerthana|     Math|   85|    Sem1|
|         2|    Rahul|  Physics|   78|    Sem1|
|         3|   Anjali|Chemistry|   92|    Sem2|
|         4|    Arjun|     Math|   88|    Sem2|
+----------+---------+---------+-----+--------+



In [5]:
df.write.partitionBy("semester").mode("overwrite").csv("student_partitioned")

In [6]:
df.filter(df.semester == "Sem1").show()

+----------+---------+-------+-----+--------+
|student_id|     name|subject|marks|semester|
+----------+---------+-------+-----+--------+
|         1|Keerthana|   Math|   85|    Sem1|
|         2|    Rahul|Physics|   78|    Sem1|
+----------+---------+-------+-----+--------+



In [7]:
df.groupBy("semester").avg("marks").show()

+--------+----------+
|semester|avg(marks)|
+--------+----------+
|    Sem1|      81.5|
|    Sem2|      90.0|
+--------+----------+



In [8]:
from pyspark.sql.functions import max

df.groupBy("semester").agg(max("marks")).show()

+--------+----------+
|semester|max(marks)|
+--------+----------+
|    Sem1|        85|
|    Sem2|        92|
+--------+----------+



PART 2: SPARK SQL CASE STUDY 1

In [9]:
data = [
    ("Laptop", "Electronics", 50000, 2),
    ("Phone", "Electronics", 20000, 5),
    ("Shirt", "Clothing", 1500, 10),
    ("Shoes", "Footwear", 3000, 4)
]

columns = ["product", "category", "price", "quantity"]

df = spark.createDataFrame(data, columns)
df.createOrReplaceTempView("sales")
df.show()

+-------+-----------+-----+--------+
|product|   category|price|quantity|
+-------+-----------+-----+--------+
| Laptop|Electronics|50000|       2|
|  Phone|Electronics|20000|       5|
|  Shirt|   Clothing| 1500|      10|
|  Shoes|   Footwear| 3000|       4|
+-------+-----------+-----+--------+



In [10]:
spark.sql("SELECT SUM(price * quantity) AS total_revenue FROM sales").show()

+-------------+
|total_revenue|
+-------------+
|       227000|
+-------------+



In [11]:
spark.sql("""
SELECT product, SUM(quantity) as total_qty
FROM sales
GROUP BY product
ORDER BY total_qty DESC
LIMIT 1
""").show()

+-------+---------+
|product|total_qty|
+-------+---------+
|  Shirt|       10|
+-------+---------+



In [12]:
spark.sql("""
SELECT category, SUM(price * quantity) as revenue
FROM sales
GROUP BY category
""").show()

+-----------+-------+
|   category|revenue|
+-----------+-------+
|Electronics| 200000|
|   Clothing|  15000|
|   Footwear|  12000|
+-----------+-------+



#SPARK SQL CASE STUDY 2

In [13]:
data = [
    (1, "login"),
    (1, "view"),
    (2, "login"),
    (1, "logout"),
    (2, "view"),
    (2, "view"),
    (3, "login")
]

columns = ["user_id", "action"]

df = spark.createDataFrame(data, columns)
df.createOrReplaceTempView("logs")
df.show()

+-------+------+
|user_id|action|
+-------+------+
|      1| login|
|      1|  view|
|      2| login|
|      1|logout|
|      2|  view|
|      2|  view|
|      3| login|
+-------+------+

